# 实验八 · Attention 与图学习

在第七次实验中，GCN 是通过节点度数来确定邻居权重。对于同一个接收节点 $i$，标准 GCN 的系数 $1/\sqrt{\hat d_i\hat d_j}$ 会给度数更大的发送节点 $j$ 更小的权重。

这次实验，我们将结合“注意力机制”，重新设计我们对节点权重的理解。具体来说，我们要研究两个问题：

1. 如果每个节点只能关注图上的邻居，我们怎样设计注意力机制？打分、归一化、聚合我们怎么做？
2. 学到的注意力权重能否预测模型对删边的敏感程度？

| 环节                    | 完成内容                       |
| ----------------------- | ------------------------------ |
| Q1 · 缩放点积与掩码     | 第 0 行的手算、非邻居扰动结果  |
| Q2 · GAT 打分           | 同一行的加性注意力权重与输出   |
| Q3 · 按边实现多头 GAT   | 稠密/按边对照、形状、梯度      |
| Q4 · Cora 模型对照      | 训练前假设、双种子验证表、曲线 |
| Q5 · 一个节点的删边对照 | 各头的权重与两次删边的预测变化 |
| Q6 · 跨节点检验         | 配对统计、结论和一条核验记录   |
| Q7 · 分层替换注意力     | 固定模型，比较准确率和预测翻转 |

> Q1 & Q2 涉及到小数据集计算检验，如果实在不想算，可以参考文末的附录……


## 0 · 环境与约定

与上次实验一样，Cora 引用关系按无向图处理，每个节点保留一个自环。四节点小图用邻接掩码展示，完整 Cora 用边表计算。行 $i$ 是接收节点，列 $j$ 是发送节点。


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from lab_utils import (
    classification_metrics,
    dense_attention,
    edge_softmax,
    fit_model,
    gcn_attention,
    load_cora,
    plot_history,
    remove_undirected_edge,
    seed_everything,
    select_validation_nodes,
    sum_messages,
    uniform_attention,
)
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(min(4, torch.get_num_threads()))
torch.set_printoptions(precision=4, sci_mode=False)


def TODO(number):
    raise NotImplementedError(f"请先完成 {number}，替换整个 TODO 表达式。")


In [ ]:
# 节点 0 与 1、2 相连，节点 3 仅保留自环。
toy_x = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [2.0, -1.0]])
toy_mask = torch.eye(len(toy_x), dtype=torch.bool)
toy_mask[0, 1:3] = True
toy_mask[1:3, 0] = True
toy_edges = torch.stack(torch.where(toy_mask))

w_query = torch.eye(2)
w_key = torch.tensor([[0.0, 1.0], [1.0, 0.0]])
w_value = torch.diag(torch.tensor([1.0, 2.0]))

## Q1 · 从全连接注意力到邻居注意力

在处理大批量的数据之前，我们先通过上面定义包含四个节点的图练练手，尝试一下注意力机制。（值得一提的是，这个处理手法我们称之为“测试”，对于复杂的代码，我们可以用一些简单易算的数据来验证）

这里我们复习一下注意力机制的基本原理：

同一组节点特征投影得到 $Q=XW_Q$、$K=XW_K$、$V=XW_V$。节点 $i$ 对节点 $j$ 的分数为 $s_{ij}=q_i^\top k_j/\sqrt{d_k}$。把允许发送消息的节点集合记作 $S_i$，则

$$
\alpha_{ij} = \frac{\exp(s_{ij})}{\sum_{k\in S_i}\exp(s_{ik})}\quad(j\in S_i),
\qquad h_i'=\sum_{j\in S_i}\alpha_{ij}v_j .
$$

全连接时 $S_i$ 包含所有节点。图掩码下 $S_i$ 只包含 $i$ 的邻居和 $i$ 自身。这里先按上述构建的图，手算接收节点 0 的分数与图掩码权重。补全代码前，先预测只改节点 3 的特征会不会改变节点 0 的输出。全连接和图掩码分别判断。

- **Q1a** 计算缩放点积矩阵，形状为 `[接收节点数, 发送节点数]`。
- **Q1b** 把屏蔽位置的分数设为 `-torch.inf`。
- **Q1c** 按发送节点维度做 softmax，使每行权重和为 1。


In [ ]:
def dot_attention(x, allowed):
    """计算点积注意力

    Args:
        x : 原矩阵
        allowed : 允许计算的位置
    """
    q, k, v = x @ w_query, x @ w_key, x @ w_value
    scores = TODO("Q1a")
    masked_scores = TODO("Q1b")
    alpha = TODO("Q1c")
    return alpha @ v, alpha


full_mask = torch.ones_like(toy_mask)
full_out, full_alpha = dot_attention(toy_x, full_mask)
masked_out, masked_alpha = dot_attention(toy_x, toy_mask)
print("节点 0 全连接权重", full_alpha[0])
print("节点 0 图掩码权重", masked_alpha[0])
print("改动节点 3 前，节点 0 全连接输出", full_out[0])
print("改动节点 3 前，节点 0 图掩码输出", masked_out[0])

assert torch.all(masked_alpha[~toy_mask] == 0)
assert torch.allclose(masked_alpha.sum(dim=1), torch.ones(len(toy_x)))
assert masked_alpha[3, 3] == 1

changed_x = toy_x.clone()
changed_x[3] = torch.tensor([5.0, 4.0])
full_changed, _ = dot_attention(changed_x, full_mask)
masked_changed, _ = dot_attention(changed_x, toy_mask)
assert not torch.allclose(full_changed[0], full_out[0])
assert torch.allclose(masked_changed[0], masked_out[0])
print("改动节点 3 后，节点 0 全连接输出", full_changed[0])
print("改动节点 3 后，节点 0 图掩码输出", masked_changed[0])

**Q1 记录 · 只看节点 0。** 运行前填预测。运行后，从两组「改动节点 3 前／后」输出各取一个二维向量。

| 规则   | 预测会变吗 | 改动前 | 改动后 |
| ------ | ---------- | ------ | ------ |
| 全连接 |            |        |        |
| 图掩码 |            |        |        |

用一句话解释图掩码下的结果，指出被改动的节点在第 0 行的权重。


## Q2 · GAT 如何给邻居打分

保留同一个邻接掩码。经典 GAT 用同一个投影 $Z=XW$ 产生消息，并用两个打分向量构造加性分数。

$$
e_{ij}=\operatorname{LeakyReLU}\bigl(a_{\rm recv}^\top z_i + a_{\rm send}^\top z_j\bigr),
\quad
\alpha_{ij}=\operatorname{softmax}_{j\in S_i}(e_{ij}),
\quad
h_i'=\sum_{j\in S_i}\alpha_{ij}z_j .
$$

LeakyReLU 在负半轴保留一个小的正斜率。

$$
\operatorname{LeakyReLU}(x) = \left\{\begin{matrix}
x & x >0\\
\alpha x & x \le 0, \alpha \in \mathbb{R}^+
\end{matrix}\right.
$$

在我们的实验小图里，取 $W=I$、$a_{\rm recv}=(1,-1)$、$a_{\rm send}=(-1,1)$，LeakyReLU 的负部斜率为 0.2。手算节点 0 对自身、节点 1 和节点 2 的分数、权重及输出。再算节点 1 对节点 0 的分数，检查负数分支。

- **Q2a** 接收项沿列广播、发送项沿行广播，先相加再做 LeakyReLU。
- **Q2b** 只在允许的邻域里做 softmax。


In [ ]:
toy_weight = torch.eye(2)
toy_a_recv = torch.tensor([1.0, -1.0])
toy_a_send = torch.tensor([-1.0, 1.0])


def dense_gat(x, allowed, weight, a_recv, a_send):
    """GAT 打分

    Args:
        x : 输入的数据
        allowed : 允许计算的位置
        weight : 权重矩阵
        a_recv : 接收矩阵
        a_send : 发送矩阵
    """
    z = x @ weight
    recv_term = z @ a_recv
    send_term = z @ a_send
    scores = TODO("Q2a")
    alpha = TODO("Q2b")
    return alpha @ z, alpha, scores


toy_out, toy_alpha, toy_scores = dense_gat(
    toy_x,
    toy_mask,
    toy_weight,
    toy_a_recv,
    toy_a_send,
)

print("节点 0 有效分数", toy_scores[0, toy_mask[0]])
print("节点 0 注意力", toy_alpha[0])
print("节点 0 聚合输出", toy_out[0])
print("节点 1 接收节点 0 的分数", toy_scores[1, 0].item())
assert torch.all(toy_alpha[~toy_mask] == 0)
assert torch.allclose(toy_alpha.sum(dim=1), torch.ones(len(toy_x)))
assert toy_alpha[3, 3] == 1
assert torch.allclose(toy_out[3], (toy_x @ toy_weight)[3])

**Q2 记录 · 区分分数和权重。** 对照输出中的「节点 0 注意力」，写出节点 1 和节点 2 的权重，并与 Q1 的图掩码权重比较。两题对这两个邻居的排序有什么区别？

- 节点 1 的权重 **\_\_**，节点 2 的权重 **\_\_**。和 Q1 的排序相比，**\_\_**。
- 用「节点 1 接收节点 0」的例子核验负斜率。激活前分数 **\_\_**，激活后分数 **\_\_**。


## Q3 · 用边表复现 Q2 的 GAT

在 Q2 中，我们先为每对节点打分，再用掩码保留边上的分数。在 Cora 中，$2708\times2708$ 的分数矩阵有 7,333,264 个位置，其中 13,264 个位置有边或自环。可以直接按边表取出发送和接收节点的特征，只为这些位置打分。

先在四节点小图上试一次。`toy_edges` 有 8 列，每列是一条消息。第二列是 `(0, 1)`，表示**节点 1 发给节点 0**。对这一列而言，`recv` 是 `0`，`send` 是 `1`。

在 `GraphLayer.forward` 补三个空。对每个头分别计算：

| 空               | 做什么                                                                                     | 节点 1 发给节点 0                                            |
| ---------------- | ------------------------------------------------------------------------------------------ | ------------------------------------------------------------ |
| **Q3a 打分**     | 每条边取接收节点的 `recv_term` 和发送节点的 `send_term`，相加后用负斜率 0.2 的 LeakyReLU。 | 两项都是 1，分数是 2。                                       |
| **Q3b 归一化**   | 用 `edge_softmax` 按 `recv` 分组，对同一个接收节点收到的所有分数做 softmax。               | 节点 0 收到的分数为 $[0,2,1]$，其中这条边权重约为 $0.6652$。 |
| **Q3c 生成消息** | 取发送节点在当前头的投影 `z`，乘这条边在该头的权重。                                       | 节点 1 发出的加权消息约为 $[0,0.6652]$。                     |

`sum_messages` 已经负责把消息按接收节点相加，再拼接各头输出。用 Q2 的同一组参数跑单头时，每条边的权重和每个节点的输出应当与 Q2 相同。双头检查中，8 条边各有 2 个头的权重，每个节点的两个 2 维输出拼成 4 维。


In [ ]:
class GraphLayer(nn.Module):
    def __init__(self, in_dim, heads, head_dim, mode):
        super().__init__()
        self.heads = heads
        self.head_dim = head_dim
        self.mode = mode
        self.proj = nn.Linear(in_dim, heads * head_dim, bias=False)
        if mode == "GAT":
            self.a_recv = nn.Parameter(torch.empty(heads, head_dim))
            self.a_send = nn.Parameter(torch.empty(heads, head_dim))
            nn.init.xavier_uniform_(self.a_recv)
            nn.init.xavier_uniform_(self.a_send)

    def forward(self, x, edges, return_attention=False):
        n = x.shape[0]
        recv, send = edges
        z = self.proj(x).reshape(n, self.heads, self.head_dim)

        if self.mode == "GAT":
            recv_term = (z * self.a_recv).sum(dim=-1)
            send_term = (z * self.a_send).sum(dim=-1)
            scores = TODO("Q3a")
            alpha = TODO("Q3b")

        elif self.mode == "等权":
            alpha = uniform_attention(recv, n, self.heads, z.dtype)
        else:
            alpha = gcn_attention(edges, n, self.heads, z.dtype)

        messages = TODO("Q3c")

        out = sum_messages(messages, recv, n).flatten(1)

        if return_attention:
            return out, alpha
        return out


class GraphClassifier(nn.Module):
    def __init__(self, in_dim, n_classes, mode):
        super().__init__()
        self.hidden = GraphLayer(in_dim, heads=2, head_dim=8, mode=mode)
        self.output = GraphLayer(16, heads=1, head_dim=n_classes, mode=mode)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x, edges):
        hidden = F.relu(self.hidden(self.dropout(x), edges))
        return self.output(self.dropout(hidden), edges)

In [ ]:
single = GraphLayer(2, 1, 2, "GAT")
with torch.no_grad():
    single.proj.weight.copy_(toy_weight.T)
    single.a_recv.copy_(toy_a_recv.unsqueeze(0))
    single.a_send.copy_(toy_a_send.unsqueeze(0))
    sparse_out, sparse_alpha = single(toy_x, toy_edges, return_attention=True)

single_alpha_dense = dense_attention(sparse_alpha, toy_edges, len(toy_x))[..., 0]
max_weight_error = (single_alpha_dense - toy_alpha).abs().max().item()
max_output_error = (sparse_out - toy_out).abs().max().item()
print(f"单头最大误差：权重 {max_weight_error:.2e}，输出 {max_output_error:.2e}")
assert torch.allclose(single_alpha_dense, toy_alpha, atol=1e-6)
assert torch.allclose(sparse_out, toy_out, atol=1e-6)

seed_everything(42)
multi = GraphLayer(2, 2, 2, "GAT")
multi_out, multi_alpha = multi(toy_x, toy_edges, return_attention=True)
assert multi_out.shape == (4, 4) and multi_alpha.shape == (8, 2)
head_sums = sum_messages(multi_alpha, toy_edges[0], len(toy_x))
print("节点 0 在两个头里的入边权重和", head_sums[0].detach())
assert torch.allclose(head_sums, torch.ones(len(toy_x), 2), atol=1e-6)
for head in range(2):
    rows = slice(2 * head, 2 * (head + 1))
    head_out, head_alpha, _ = dense_gat(
        toy_x,
        toy_mask,
        multi.proj.weight[rows].T,
        multi.a_recv[head],
        multi.a_send[head],
    )
    assert torch.allclose(multi_out[:, rows], head_out, atol=1e-6)
    assert torch.allclose(multi_alpha[:, head], head_alpha[toy_mask], atol=1e-6)

multi_out[0].square().sum().backward()
for name in ("proj.weight", "a_send"):
    grad = dict(multi.named_parameters())[name].grad
    assert grad is not None and torch.isfinite(grad).all() and grad.norm() > 0
print("通过：单头稠密/按边对照、逐头拼接、归一化及梯度检查")

**Q3 记录 · 留下一条可复查的证据。**

- 从「单头最大误差」抄下权重误差 **\_\_**、节点输出误差 **\_\_**。说明为什么这里能把 Q2 当作参照。
- 节点 0 的两个头各自求和得到 **\_\_**。如果误按发送节点分组，这个检查在验证什么条件？


## Q4 · 在 Cora 上比较三种聚合规则

这里沿用实验七的 Cora 数据、特征行归一化和种子 42 的固定划分。训练每类 20 个标签，共 140 个。用 500 个验证节点选轮次并比较模型，保留 2,068 个测试节点。全图结构和特征仍参与前向计算。

| 模型 | 节点 $i$ 从邻居 $j$ 接收消息的权重          |
| ---- | ------------------------------------------- |
| GCN  | $1/\sqrt{\hat d_i\hat d_j}$，使用含自环度数 |
| 等权 | $1/\hat d_i$，同一接收节点的所有邻居等权    |
| GAT  | 特征打分后在接收节点邻域内做 softmax        |

三组都使用 `1433 → 16 → 7`、ReLU、特征 dropout 0.5 和相同优化预算。同一个种子下两层的投影从相同参数开始。GAT 额外学习 46 个打分参数。固定规则与可学习打分都用同一批边。每组独立训练并按验证准确率选轮次，准确率并列时优先验证损失低、轮次早的快照。

**运行前各写一句预测。** 比较时，每次都在同一个模型种子下看两种聚合规则。

- 把 GCN 改成邻居等权后，我预期验证准确率 **\_\_**，理由是 **\_\_**。
- 把邻居等权改成 GAT 后，我预期验证准确率 **\_\_**，理由是 **\_\_**。


In [ ]:
data = load_cora()
x, y, edges = data["x"], data["y"], data["edges"]
class_names = data["class_names"]
split = data["split"]
train_idx, val_idx = split["train"], split["val"]

assert x.shape == (2708, 1433) and edges.shape == (2, 13264)
assert [len(split[name]) for name in ("train", "val", "test")] == [140, 500, 2068]
assert torch.bincount(y[train_idx]).tolist() == [20] * 7
print("节点特征、边形状", tuple(x.shape), tuple(edges.shape))
print("训练/验证/保留测试节点数", len(train_idx), len(val_idx), len(split["test"]))

seeds = (42, 2026)
modes = ("GCN", "等权", "GAT")
config = {"epochs": 200, "lr": 0.01, "weight_decay": 0.0005, "log_every": 0}
runs = {}
for seed in seeds:
    seed_everything(seed)
    template = GraphClassifier(x.shape[1], len(class_names), "GCN")
    initial_projections = {
        name: p.detach().clone() for name, p in template.named_parameters()
    }
    for mode in modes:
        seed_everything(seed)
        model = GraphClassifier(x.shape[1], len(class_names), mode)
        model.load_state_dict(initial_projections, strict=(mode != "GAT"))
        # 模型构造消耗的随机数不同，从同一个随机状态开始训练。
        seed_everything(seed)
        run = fit_model(
            model, x, edges, train_idx, y[train_idx], val_idx, y[val_idx], **config
        )
        with torch.no_grad():
            metrics = classification_metrics(model(x, edges)[val_idx], y[val_idx])
        runs[seed, mode] = run
        print(
            f"{mode:4s} seed={seed:4d} epoch={run['best_epoch']:3d}  "
            f"Val Acc={metrics['accuracy']:.2%}  Val Macro-F1={metrics['macro_f1']:.2%}  "
            f"参数={run['parameters']['trainable']:,}  用时={run['elapsed_seconds']:.2f}s"
        )

print()
print("验证准确率的均值与标准差（两个种子，ddof=0）")
for mode in modes:
    accuracies = np.array([runs[seed, mode]["best_val_acc"] for seed in seeds])
    print(f"{mode:4s}  {accuracies.mean():.2%} ± {100 * accuracies.std():.2f} 个百分点")

gat_history = runs[seeds[0], "GAT"]["history"]
best_gat = gat_history[runs[seeds[0], "GAT"]["best_epoch"] - 1]
print("\n种子 42 的 GAT，比较最佳轮和末轮")
for row in (best_gat, gat_history[-1]):
    print(
        f"第 {row['epoch']:3d} 轮  训练准确率 {row['train_acc']:.2%}  "
        f"验证准确率 {row['val_acc']:.2%}"
    )

fig = plot_history(
    {
        {"GCN": "GCN", "等权": "Mean", "GAT": "GAT"}[mode]: runs[seeds[0], mode]
        for mode in modes
    }
)
plt.show()

**Q4 记录 · 比同种子，再读曲线。** 从六行训练结果中取 `Val Acc`。差值统一用百分点，例如 79.00% − 78.50% = 0.50 个百分点。

| 指标           | 种子 42 | 种子 2026 |
| -------------- | ------- | --------- |
| GCN 验证准确率 |         |           |
| 等权验证准确率 |         |           |
| GAT 验证准确率 |         |           |
| 等权 − GCN     |         |           |
| GAT − 等权     |         |           |

1. 两个种子的差值方向是否一致？你的哪条预测得到支持，哪条需要修正？
2. 看种子 42 的 GAT 曲线，结合「最佳轮和末轮」的输出。最佳轮为第 **\_\_** 轮。到末轮时，训练准确率从 **\_\_** 到 **\_\_**，验证准确率从 **\_\_** 到 **\_\_**。按本实验的选轮规则，为什么保留的是最佳轮？


## Q5 · 同一个节点，删高权重边和低权重边

从数学抽象来讲，我们总说，边的权重越高，意味着这条边包含着更多的信息，有助于我们进行更精确的预测。那么，从实际结果来看，这些边及其权重，到底会怎样影响我们的模型预测结果呢？

这里我们取用和实验七种一样的配置。我们取出某一个节点，画出第一层各头收到的权重。排除自环，用第一层两头的平均权重选择最高和最低的入边。若分数相同，则按发送节点下标排序后取两端。两次都从原图出发，各删除一条无向关系的两个方向，然后再对全图重新前向。运行前预测：删高权重边和删低权重边，哪个影响更大？


In [ ]:
gat = runs[seeds[0], "GAT"]["model"].eval()
probe_nodes = select_validation_nodes(edges, val_idx, limit=20)
probe = probe_nodes[len(probe_nodes) // 2]

with torch.no_grad():
    _, first_alpha = gat.hidden(x, edges, return_attention=True)
    original_probs = gat(x, edges).softmax(dim=1)


def neighbor_edges(node):
    incoming = (edges[0] == node) & (edges[1] != node)
    return torch.where(incoming)[0].tolist()


def rank_neighbors(node):
    return sorted(
        neighbor_edges(node),
        key=lambda e: (float(first_alpha[e].mean()), int(edges[1, e])),
    )


recv_edges = torch.where(edges[0] == probe)[0]
tick_labels = [
    str(sender) if sender != probe else f"{sender} (self)"
    for sender in edges[1, recv_edges].tolist()
]
positions = np.arange(len(recv_edges))
fig, ax = plt.subplots(figsize=(8, 3.4), constrained_layout=True)
for head in range(first_alpha.shape[1]):
    ax.bar(
        positions + (head - 0.5) * 0.35,
        first_alpha[recv_edges, head].numpy(),
        width=0.35,
        label=f"Head {head + 1}",
    )
ax.set(
    xticks=positions,
    xticklabels=tick_labels,
    xlabel="Sending node",
    ylabel="Attention weight",
    title=f"First-layer attention received by node {probe}",
)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
plt.show()
print("发送节点  各头注意力权重")
for name, weights in zip(tick_labels, first_alpha[recv_edges].tolist()):
    print(f"{name:>10s}  " + "  ".join(f"{value:.5f}" for value in weights))


@torch.no_grad()
def ablate_one(node, edge_position):
    sender = int(edges[1, edge_position])
    reduced_edges = remove_undirected_edge(edges, node, sender)
    assert reduced_edges.shape[1] == edges.shape[1] - 2
    base = original_probs[node]
    original_class = int(base.argmax())
    after = gat(x, reduced_edges)[node].softmax(dim=0)

    # Q5a：保留原预测类别，计算删边后同一类别的概率（0 到 1）。
    after_original_prob = TODO("Q5a")

    return {
        "sender": sender,
        "mean_alpha": float(first_alpha[edge_position].mean()),
        "before_prob": float(base[original_class]),
        "after_prob": float(after_original_prob),
        "before_class": original_class,
        "after_class": int(after.argmax()),
    }


def compare_removals(node):
    ranked = rank_neighbors(node)
    assert len(ranked) >= 2
    return {
        kind: ablate_one(node, edge)
        for kind, edge in [("高权重", ranked[-1]), ("低权重", ranked[0])]
    }


print(f"接收节点 {probe}，外部邻居数 {len(neighbor_edges(probe))}")
for kind, result in compare_removals(probe).items():
    change = 100 * (result["after_prob"] - result["before_prob"])
    flipped = "是" if result["after_class"] != result["before_class"] else "否"
    print(
        f"{kind}  发送节点 {result['sender']:4d}  平均权重={result['mean_alpha']:.4f}  "
        f"{class_names[result['before_class']]} → {class_names[result['after_class']]}  "
        f"原类别概率 {result['before_prob']:.2%} → {result['after_prob']:.2%}  "
        f"变化 {change:+.3f} 个百分点  翻转={flipped}"
    )

**Q5 记录 · 固定同一个节点和原预测类别。** 从「接收节点」和两行删边结果取数。原预测类别是删边前概率最高的类别，两次都追踪它。变化统一用「删后减删前」，单位为百分点。

- 接收节点编号 **\_\_**，外部邻居数 **\_\_**，原预测类别 **\_\_**。

| 删除哪条关系 | 发送节点编号 | 两头平均权重 | 原类别概率，删前 → 删后 | 变化（百分点） | 预测翻转？ |
| ------------ | ------------ | ------------ | ----------------------- | -------------- | ---------- |
| 高权重边     |              |              |                         |                |            |
| 低权重边     |              |              |                         |                |            |

先比表里两个变化的绝对值，判断哪条边的影响更大。再看「各头注意力权重」表，找两个外部邻居核对两头对它们的相对排序。删边后剩余邻居的权重会发生什么变化？


## Q6 · 在多个节点上做配对比较

在 Q5 固定的名单上重复对照。对每个节点各得两次干预结果，比较删高、低权重边导致的**原预测类别概率变化的绝对值**。绝对值衡量变化大小，概率升高或降低可从 Q5 的有符号变化读出。

- **Q6a** 用 `before_prob` 和 `after_prob` 计算绝对变化，统一换算成百分点。
- 报告两类删边的平均绝对变化、各自的翻转数，以及多少节点的高权重删边变化更大。把差异与 Q5 的例子对照。

配对图里一个点对应一个验证节点。横轴是删低权重边的绝对变化，纵轴是删高权重边的绝对变化。对角线上方表示删高权重边的影响更大，下方表示删低权重边的影响更大。


In [ ]:
def absolute_change_pp(result):
    """计算概率变化的绝对值"""
    return TODO("Q6a")


paired = {node: compare_removals(node) for node in probe_nodes}
high_changes = np.array(
    [absolute_change_pp(pair["高权重"]) for pair in paired.values()]
)
low_changes = np.array([absolute_change_pp(pair["低权重"]) for pair in paired.values()])
tied = np.isclose(high_changes, low_changes, atol=1e-5, rtol=0)

print("节点编号  外部度数  删高权重边  删低权重边（绝对变化，百分点）")
for node, high, low in zip(paired, high_changes, low_changes):
    print(f"{node:8d}  {len(neighbor_edges(node)):8d}  {high:10.3f}  {low:10.3f}")
print()
print(f"配对节点数 {len(paired)}")
for kind, changes in [("高权重", high_changes), ("低权重", low_changes)]:
    flips = sum(
        pair[kind]["after_class"] != pair[kind]["before_class"]
        for pair in paired.values()
    )
    print(f"{kind}  平均绝对变化={changes.mean():.3f} 个百分点  翻转数={flips}")
print(f"平均配对差（高−低） {(high_changes - low_changes).mean():+.3f} 个百分点")

high_larger = int(((high_changes > low_changes) & ~tied).sum())
low_larger = int(((low_changes > high_changes) & ~tied).sum())
print(
    f"高权重变化更大 {high_larger}，低权重变化更大 {low_larger}，近似相等 {tied.sum()}"
)

limit = max(float(high_changes.max()), float(low_changes.max()), 0.1) * 1.08
fig, ax = plt.subplots(figsize=(5.1, 4.8), constrained_layout=True)
ax.plot([0, limit], [0, limit], color="0.55", linestyle="--", label="Equal change")
ax.scatter(
    low_changes, high_changes, color="tab:blue", alpha=0.8, label="Validation node"
)
probe_position = probe_nodes.index(probe)
ax.scatter(
    low_changes[probe_position],
    high_changes[probe_position],
    color="tab:orange",
    s=60,
    label=f"Q5 node {probe}",
    zorder=3,
)
ax.set(
    xlim=(0, limit),
    ylim=(0, limit),
    aspect="equal",
    xlabel="Remove low-weight edge (pp)",
    ylabel="Remove high-weight edge (pp)",
)
ax.legend()
plt.show()

**Q6 记录 · 用总体数字和一个节点回答。**

| 删边方式 | 平均绝对变化（百分点） | 预测翻转数 |
| -------- | ---------------------- | ---------- |
| 高权重边 |                        |            |
| 低权重边 |                        |            |

- 共 **\_\_** 对。高权重边的变化更大 **\_\_** 对，低权重边的变化更大 **\_\_** 对，近似相等 **\_\_** 对。
- 从逐节点输出中找一个能补充总体均值的例子，优先选对角线下方且与 Q5 不同的节点。节点编号 **\_\_**，删高权重边的绝对变化 **\_\_**，删低权重边的绝对变化 **\_\_**。
- 用两句话回答「高权重是否通常对应更大的删边影响？」第一句引用均值和配对计数，第二句用所选节点说明结论的适用范围。


## Q7 · 推理时把注意力换成等权

Q4 中，GAT 和等权模型是各自训练的。这里我们尝试一个新的玩法：**同一个已经训练好的 GAT，如果只在预测时把某一层换成等权，它还会给出同样的预测吗？**

我们直接复制 Q4 已经训练好的两个 GAT 结果，每次都从原型中复制一份来处理。我们比较四层内容：原模型、仅隐藏层等权、仅输出层等权、两层等权。

用 Q4 已经训练好的两个 GAT 快照。每次从原模型复制一份，保留投影参数、同一张图和同一批验证节点。分别比较原模型、仅隐藏层等权、仅输出层等权、两层等权。替换时，每条入边的系数变成 $1/\hat d_i$。如果只换隐藏层，输出层仍按 GAT 规则用新的隐藏表示重新打分。

**运行前预测。** 只换一层时，你认为哪一层会让更多验证节点改判？写一句理由。


In [ ]:
from copy import deepcopy


@torch.no_grad()
def predict_with_equal_layers(trained, layers):
    model = deepcopy(trained).eval()
    for layer in layers:
        getattr(model, layer).mode = "等权"
    return model(x, edges)[val_idx].softmax(dim=1)


variants = {
    "原 GAT": (),
    "仅隐藏层等权": ("hidden",),
    "仅输出层等权": ("output",),
    "两层等权": ("hidden", "output"),
}
val_labels = y[val_idx]
for seed in seeds:
    trained = runs[seed, "GAT"]["model"]
    base_probs = predict_with_equal_layers(trained, ())
    base_class = base_probs.argmax(dim=1)
    print(f"种子 {seed}，验证节点 {len(val_idx)} 个")
    for name, layers in variants.items():
        probs = predict_with_equal_layers(trained, layers) if layers else base_probs
        predicted = probs.argmax(dim=1)
        accuracy = float((predicted == val_labels).float().mean())
        changed = torch.where(predicted != base_class)[0]
        print(f"{name}  验证准确率={accuracy:.2%}  翻转={len(changed)}")
        if len(changed):
            i = int(changed[0])
            label_before = int(base_class[i])
            label_after = int(predicted[i])
            print(
                f"  首个翻转节点 {int(val_idx[i])}，"
                f"{class_names[label_before]} → {class_names[label_after]}，"
                f"真实类别 {class_names[int(val_labels[i])]}，"
                f"原类别概率 {base_probs[i, label_before]:.2%} → {probs[i, label_before]:.2%}"
            )
    assert trained.hidden.mode == trained.output.mode == "GAT"
    print()

**Q7 记录 · 同时看全体验证准确率和个体预测。** 每格记录「验证准确率／翻转数」，翻转指预测类别相对同种子原 GAT 发生变化。

| 预测时用的聚合规则 | 种子 42 | 种子 2026 |
| ------------------ | ------- | --------- |
| 原 GAT             |         |           |
| 仅隐藏层等权       |         |           |
| 仅输出层等权       |         |           |
| 两层等权           |         |           |

1. 对照运行前的预测。只换一层时，哪个层的翻转数较大？两个种子的结果是否一致？
2. 找一组「准确率相同，但有节点翻转」的结果，引用输出中的一个节点，说明总体准确率相同为什么仍可有不同预测。如果没有这种组合，任选一个有翻转的条件，描述一个节点的变化。所有条件都没有翻转时，记录这一结果。
3. 对照 Q4 的独立训练等权模型和 Q7 的「两层等权」。两组都使用等权传播，投影参数分别是怎么得到的？


## 附录

如果实在不想算，答案放在这里……

### 四节点小图的实际数值

节点按 $0,1,2,3$ 排列。在邻接矩阵、分数矩阵和注意力矩阵中，行 $i$ 是接收节点，列 $j$ 是发送节点。以下小数保留四位。

#### Q0 · 节点特征、邻接掩码与边表

`toy_x` 是节点特征矩阵。`toy_mask` 是布尔邻接掩码，用 $1$ 表示 `True`，用 $0$ 表示 `False`。`toy_edges` 每列是一条消息边，第一行是接收节点，第二行是发送节点。

$$
X=\begin{pmatrix}
1&0\\
0&1\\
1&1\\
2&-1
\end{pmatrix},
\qquad
A=\begin{pmatrix}
1&1&1&0\\
1&1&0&0\\
1&0&1&0\\
0&0&0&1
\end{pmatrix},
\qquad
\texttt{toy\_edges}=\begin{pmatrix}
0&0&0&1&1&2&2&3\\
0&1&2&0&1&0&2&3
\end{pmatrix}.
$$

加到注意力分数上的掩码 $M$ 由 $A$ 得到。允许的位置加 $0$，屏蔽的位置加 $-\infty$。

$$
M=\begin{pmatrix}
0&0&0&-\infty\\
0&0&-\infty&-\infty\\
0&-\infty&0&-\infty\\
-\infty&-\infty&-\infty&0
\end{pmatrix}.
$$

#### Q1 · 缩放点积注意力

`w_query`、`w_key`、`w_value` 分别对应以下三个投影矩阵。

$$
W_Q=\begin{pmatrix}1&0\\0&1\end{pmatrix},
\quad
W_K=\begin{pmatrix}0&1\\1&0\end{pmatrix},
\quad
W_V=\begin{pmatrix}1&0\\0&2\end{pmatrix}.
$$

投影得到 $Q=X$ 以及下面的 $K$ 和 $V$。缩放分数为 $S=QK^\top/\sqrt{2}$。

$$
K=\begin{pmatrix}
0&1\\1&0\\1&1\\-1&2
\end{pmatrix},
\qquad
V=\begin{pmatrix}
1&0\\0&2\\1&2\\2&-2
\end{pmatrix},
\qquad
S=\frac{1}{\sqrt{2}}\begin{pmatrix}
0&1&1&-1\\
1&0&1&2\\
1&1&2&1\\
-1&2&1&-4
\end{pmatrix}.
$$

`full_mask` 是全 `True` 的 $4\times4$ 矩阵。全连接注意力对 $S$ 逐行做 softmax，图掩码注意力对 $S+M$ 逐行做 softmax。两者都用所得权重乘 $V$。

| 接收节点 | `full_alpha`                    | `masked_alpha`             | `full_out`         | `masked_out`       |
| -------- | ------------------------------- | -------------------------- | ------------------ | ------------------ |
| 0        | $[0.1802,0.3655,0.3655,0.0889]$ | $[0.1978,0.4011,0.4011,0]$ | $[0.7234,1.2842]$  | $[0.5989,1.6044]$  |
| 1        | $[0.2212,0.1091,0.2212,0.4486]$ | $[0.6698,0.3302,0,0]$      | $[1.3395,-0.2367]$ | $[0.6698,0.6605]$  |
| 2        | $[0.1989,0.1989,0.4034,0.1989]$ | $[0.3302,0,0.6698,0]$      | $[1.0000,0.8067]$  | $[1.0000,1.3395]$  |
| 3        | $[0.0737,0.6145,0.3030,0.0088]$ | $[0,0,0,1]$                | $[0.3943,1.8174]$  | $[2.0000,-2.0000]$ |

`changed_x` 将 $X$ 的第 $3$ 行从 $[2,-1]$ 改为 $[5,4]$。重新计算得到

$$
\texttt{full\_changed}\approx
\begin{pmatrix}
3.9873&6.5285\\
4.4609&7.1264\\
4.9410&7.9098\\
5.0000&8.0000
\end{pmatrix},
\qquad
\texttt{masked\_changed}\approx
\begin{pmatrix}
0.5989&1.6044\\
0.6698&0.6605\\
1.0000&1.3395\\
5.0000&8.0000
\end{pmatrix}.
$$

图掩码下，节点 $0,1,2$ 都收不到节点 $3$ 的消息，所以它们的输出保持原值。节点 $3$ 的权重始终全部落在自环上，输出随自身的值向量从 $[2,-2]$ 变为 $[5,8]$。

#### Q2 · 同一张图上的加性注意力

`toy_weight` 是 $I_2$，所以投影后的 $Z=X$。`toy_a_recv` 为 $[1,-1]$，`toy_a_send` 为 $[-1,1]$。`toy_scores` 是 LeakyReLU 之后、加图掩码之前的分数矩阵。对 `toy_scores` 加 $M$ 后逐行做 softmax，得到 `toy_alpha`，再乘 $Z$ 得到 `toy_out`。

| 接收节点 | `toy_scores`         | `toy_alpha`                | `toy_out`          |
| -------- | -------------------- | -------------------------- | ------------------ |
| 0        | $[0,2,1,-0.4]$       | $[0.0900,0.6652,0.2447,0]$ | $[0.3348,0.9100]$  |
| 1        | $[-0.4,0,-0.2,-0.8]$ | $[0.4013,0.5987,0,0]$      | $[0.4013,0.5987]$  |
| 2        | $[-0.2,1,0,-0.6]$    | $[0.4502,0,0.5498,0]$      | $[1.0000,0.5498]$  |
| 3        | $[2,4,3,0]$          | $[0,0,0,1]$                | $[2.0000,-1.0000]$ |

以节点 $0$ 为例，点积注意力给节点 $1$ 和 $2$ 相同的权重 $0.4011$。换成加性打分后，两者的权重分别为 $0.6652$ 和 $0.2447$。两次计算允许发送消息的节点集合相同。
